# 20. PM10 데이터를 LSTM 입력 모양으로 만들기

이번 장에서는 PM10 시계열 데이터를 LSTM에 넣을 수 있는 모양으로 바꿉니다.

핵심 모양:

```text
(samples, timesteps, features)
```

예를 들어 `(1000, 24, 1)`은 샘플 1000개, 각 샘플은 과거 24시간, 각 시간의 특성은 PM10 하나라는 뜻입니다.

## 1. 라이브러리 불러오기

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# MinMaxScaler는 숫자를 0~1 범위로 변환할 때 자주 사용합니다.
from sklearn.preprocessing import MinMaxScaler

## 2. 작은 숫자로 window 이해하기

먼저 PM10 데이터가 아니라 아주 작은 숫자로 원리를 봅니다.

window가 3이면 과거 3개 값을 보고 다음 값을 맞힙니다.

In [ ]:
toy_values = np.array([10, 20, 30, 40, 50, 60])
window_size = 3

X_toy = []
y_toy = []

for i in range(len(toy_values) - window_size):
    # i부터 i+window_size 직전까지가 입력입니다.
    X_toy.append(toy_values[i:i + window_size])
    
    # 그 바로 다음 값이 정답입니다.
    y_toy.append(toy_values[i + window_size])

X_toy = np.array(X_toy)
y_toy = np.array(y_toy)

print("X_toy:")
print(X_toy)
print("\ny_toy:")
print(y_toy)
print("\nX_toy shape:", X_toy.shape)
print("y_toy shape:", y_toy.shape)

## 3. LSTM용 3차원 모양으로 바꾸기

위의 `X_toy`는 아직 `(샘플 수, 시간 길이)` 모양입니다.

LSTM은 `(샘플 수, 시간 길이, 특성 수)`를 기대하므로 마지막 차원을 하나 추가합니다.

In [ ]:
# reshape()는 배열의 모양을 바꿉니다.
# 여기서는 features=1 차원을 추가합니다.
X_toy_lstm = X_toy.reshape((X_toy.shape[0], X_toy.shape[1], 1))

print("LSTM 입력용 X_toy_lstm shape:", X_toy_lstm.shape)
print(X_toy_lstm)

## 4. PM10 데이터 읽기

In [ ]:
DATA_PATH = Path(r"C:\work\dataset\seoul_pm10.csv")

# 한국어 CSV이므로 cp949 인코딩으로 읽습니다.
df = pd.read_csv(DATA_PATH, encoding="cp949")

# 문자열 날짜를 datetime 타입으로 변환합니다.
df["date"] = pd.to_datetime(df["date"], errors="coerce")

print("데이터 shape:", df.shape)
df.head()

## 5. 강남구 PM10만 선택하고 정리하기

In [ ]:
TARGET_AREA = "강남구"

area_df = df[df["area"] == TARGET_AREA].copy()
area_df = area_df.sort_values("date")

# PM10 결측치는 앞 시간 값으로 채웁니다.
area_df["pm10_filled"] = area_df["pm10"].ffill()

# 그래도 맨 앞에 결측치가 남을 수 있으므로 제거합니다.
area_df = area_df.dropna(subset=["pm10_filled"])

print("선택 지역:", TARGET_AREA)
print("정리 후 shape:", area_df.shape)
print("날짜 범위:", area_df["date"].min(), "~", area_df["date"].max())

## 6. 시간 순서대로 train/validation 나누기

시계열 데이터에서는 무작위로 섞지 않습니다.

앞쪽 80%를 학습용, 뒤쪽 20%를 검증용으로 둡니다.

In [ ]:
pm10_values = area_df["pm10_filled"].values.reshape(-1, 1)

# 전체 길이의 80% 지점을 나눔 기준으로 잡습니다.
split_index = int(len(pm10_values) * 0.8)

train_values = pm10_values[:split_index]
val_values = pm10_values[split_index:]

print("전체 길이:", len(pm10_values))
print("train 길이:", len(train_values))
print("validation 길이:", len(val_values))

## 7. MinMaxScaler 적용

LSTM에는 보통 값을 0~1 범위로 맞춰 넣는 경우가 많습니다.

중요한 규칙:

```text
scaler는 train 데이터에만 fit한다.
validation 데이터에는 transform만 한다.
```

In [ ]:
scaler = MinMaxScaler()

# fit_transform()은 train 데이터에서 최솟값/최댓값을 배우고 변환합니다.
train_scaled = scaler.fit_transform(train_values)

# transform()은 train에서 배운 기준으로 validation 데이터를 변환합니다.
val_scaled = scaler.transform(val_values)

print("train_scaled min/max:", train_scaled.min(), train_scaled.max())
print("val_scaled min/max:", val_scaled.min(), val_scaled.max())

## 8. window 데이터셋 생성 함수 만들기

In [ ]:
def make_window_dataset(values, window_size):
    """과거 window_size개 값을 X로, 그 다음 값을 y로 만드는 함수입니다."""
    
    X = []
    y = []
    
    for i in range(len(values) - window_size):
        # 과거 window_size개 값입니다.
        X.append(values[i:i + window_size])
        
        # 바로 다음 시점의 값입니다.
        y.append(values[i + window_size])
    
    return np.array(X), np.array(y)

## 9. PM10 window 데이터 만들기

window size를 24로 두면 과거 24시간을 보고 다음 시간 PM10을 예측하는 문제가 됩니다.

In [ ]:
WINDOW_SIZE = 24

X_train, y_train = make_window_dataset(train_scaled, WINDOW_SIZE)
X_val, y_val = make_window_dataset(val_scaled, WINDOW_SIZE)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_val shape:", X_val.shape)
print("y_val shape:", y_val.shape)

## 10. shape 읽기

`X_train shape`가 예를 들어 `(6983, 24, 1)`이라면 이렇게 읽습니다.

```text
학습 샘플이 6983개 있다.
각 샘플은 과거 24시간을 포함한다.
각 시간에는 PM10 특성 1개가 있다.
```

In [ ]:
sample_count, timesteps, features = X_train.shape

print("samples:", sample_count)
print("timesteps:", timesteps)
print("features:", features)

## 11. 첫 번째 샘플 확인하기

In [ ]:
print("첫 번째 입력 X:")
print(X_train[0].ravel())

print("\n첫 번째 정답 y:")
print(y_train[0])

## 12. 원래 PM10 값으로 되돌려 보기

스케일링된 값은 0~1 범위라 실제 PM10 값이 바로 보이지 않습니다.

`inverse_transform()`을 사용하면 원래 단위로 되돌릴 수 있습니다.

In [ ]:
# y_train[0]은 shape이 (1,)이므로 inverse_transform에 넣기 위해 2차원으로 바꿉니다.
first_y_original = scaler.inverse_transform(y_train[0].reshape(1, -1))

print("첫 번째 정답의 원래 PM10 값:", first_y_original[0][0])

## 13. 입력 window와 정답을 그림으로 보기

In [ ]:
first_x_original = scaler.inverse_transform(X_train[0].reshape(-1, 1)).ravel()
first_y_original = scaler.inverse_transform(y_train[0].reshape(1, -1))[0][0]

plt.figure(figsize=(8, 4))
plt.plot(range(WINDOW_SIZE), first_x_original, marker="o", label="input 24 hours")
plt.scatter(WINDOW_SIZE, first_y_original, color="red", label="target next hour")
plt.title("One Window Sample")
plt.xlabel("Time step")
plt.ylabel("PM10")
plt.legend()
plt.tight_layout()
plt.show()

## 14. 이번 장 정리

이번 장에서 배운 핵심은 다음과 같습니다.

```text
1. LSTM 입력은 (samples, timesteps, features) 모양이다.
2. window는 과거 몇 개 시점을 볼지 정하는 길이다.
3. PM10 하나만 쓰면 features는 1이다.
4. 시계열 train/validation split은 시간 순서를 지켜야 한다.
5. scaler는 train 데이터에만 fit해야 한다.
```

## 과제

1. `WINDOW_SIZE`를 6, 24, 72로 바꾸면 각각 어떤 의미인지 설명해보세요.
2. `X_train.shape`를 자기 말로 읽어보세요.
3. `inverse_transform()`이 왜 필요한지 설명해보세요.
4. 다음 장에서 LSTM 모델이 `X_train`을 어떻게 사용할지 예상해보세요.